In [ ]:
# @title Chapter 6 第3〜4回のコード完成例

# 学習用ベース → https://colab.research.google.com/github/ec22s/colab-ikinari-python/blob/main/base/base_chapter_6_3-4.ipynb

# (1) 省略 (ベースのままで動く)
# (2) 省略 (動画ファイル名を差し替えるだけ)
# (3) ここに転記 (1回だけのライブラリインストール)

import subprocess
subprocess.run(["pip", "install", "ultralytics"])

# (4) 省略 (https://colab.research.google.com/github/ec22s/colab-ikinari-python/blob/main/completed/completed_chapter_6_2.ipynb (10) の動画ファイル名を差し替え、必要に応じ人数表示のサイズを調整するだけ)
# (5)以降は各セルを参照

In [2]:
# @title 事前準備：Colabで画像表示する関数の読み込み

import requests

branch = "https://github.com/ec22s/colab-ikinari-python/raw/refs/heads/main"
paths = [ "util/colab_imshow.py" ]
for path in paths:
  exec(requests.get(f"{branch}/{path}", allow_redirects=True).content)

In [ ]:
# @title (5) 顔の領域を抽出 (本 6-5-1)

# VideoCaptureメソッドに渡す動画ファイル名は各自で入力

# 画像上部の人数表示の大きさを調整する場合、変える数値パラメータは下記3箇所
# extension_height = 40
# font_scale = 1
# img_annotated = text_overwrite_to_image(..., (10, 25))

# 顔が検出されない(にくい)時は
# face_cascade.detectMultiScale のパラメータを調整 (本 p.200 参照)

from ultralytics import YOLO
import cv2
import numpy as np

cap = cv2.VideoCapture("movie.mp4")

def text_overwrite_to_image(img, text, position):
  """テキストを画像に描画する関数"""

  # 画像の横サイズを取得
  img_width = img.shape[1]

  # 黒い四角形（拡張領域）を作成
  extension_height = 40
  black_rectangle = np.zeros((extension_height, img_width, 3), dtype=np.uint8)

  # 元の画像と黒い四角形を結合
  extended_img = np.vstack((black_rectangle, img))

  # 文字の設定
  font = cv2.FONT_HERSHEY_SIMPLEX
  font_scale = 1
  font_color = (255, 255, 255)
  font_thickness = 2

  # 画像にテキストを上書き
  cv2.putText(extended_img,
              text=text,
              org=position,
              fontFace=font,
              fontScale=font_scale,
              color=font_color,
              thickness=font_thickness)

  return extended_img

def detect_faces(frame, person_box, face_cascade):
  """顔を検出・描画する関数"""

  # 物体検出領域から顔を検出
  x1, y1, x2, y2 = int(person_box[0]), int(person_box[1]), int(person_box[2]), int(person_box[3])
  roi_gray = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
  faces = face_cascade.detectMultiScale(roi_gray, 1.1, 4)

  # 顔検出結果をバウンディングボックスで描画
  for (fx, fy, fw, fh) in faces:
    roi_face_gray = roi_gray[fy:fy+fh, fx:fx+fw]
    face_top_left = (x1 + fx, y1 + fy)
    face_bottom_right = (x1 + fx + fw, y1 + fy + fh)
    cv2.rectangle(frame, face_top_left, face_bottom_right, (0, 255, 0), 2)

  return frame

# モデルを設定
model = YOLO("yolov8n.pt")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

try:
  while cap.isOpened():
    # フレームを抽出
    ret, frame = cap.read()

    if ret:
      # 物体検出
      results = model(frame, verbose=False, classes=0)
      img_annotated = results[0].plot()

      # 分析（検出数をカウント）
      class_list = results[0].boxes.cls
      count_class = len(class_list)

      # 物体検出部分の領域を取得
      person_boxes = results[0].boxes.xyxy

      # 顔検出
      for box in person_boxes:
        img_annotated = detect_faces(img_annotated, box, face_cascade)

      # 画像に検出数を表示
      img_annotated = text_overwrite_to_image(img_annotated, f"person={count_class}", (10, 25))
      colab_imshow("png", img_annotated)

except KeyboardInterrupt:
  print("中止しました")

In [ ]:
# @title (6) 笑顔を検出 (本 6-5-2)

# VideoCaptureメソッドに渡す動画ファイル名は各自で入力

# 画像上部の人数表示の大きさを調整する場合、変える数値パラメータは下記3箇所
# extension_height = 40
# font_scale = 1
# img_annotated = text_overwrite_to_image(..., (10, 25))

# 顔・笑顔が検出されない(にくい)時は
# face(smile)_cascade.detectMultiScale のパラメータを調整 (本 p.200 参照)

from ultralytics import YOLO
import cv2
import numpy as np

cap = cv2.VideoCapture("movie.mp4")

def text_overwrite_to_image(img, text, position):
  """テキストを画像に描画する関数"""

  # 画像の横サイズを取得
  img_width = img.shape[1]

  # 黒い四角形（拡張領域）を作成
  extension_height = 40
  black_rectangle = np.zeros((extension_height, img_width, 3), dtype=np.uint8)

  # 元の画像と黒い四角形を結合
  extended_img = np.vstack((black_rectangle, img))

  # 文字の設定
  font = cv2.FONT_HERSHEY_SIMPLEX
  font_scale = 1
  font_color = (255, 255, 255)
  font_thickness = 2

  # 画像にテキストを上書き
  cv2.putText(extended_img,
              text=text,
              org=position,
              fontFace=font,
              fontScale=font_scale,
              color=font_color,
              thickness=font_thickness)

  return extended_img

def detect_faces(frame, person_box, face_cascade, smile_cascade):
  """顔と笑顔を検出・描画する関数"""

  # 物体検出領域から顔を検出
  x1, y1, x2, y2 = int(person_box[0]), int(person_box[1]), int(person_box[2]), int(person_box[3])
  roi_gray = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
  faces = face_cascade.detectMultiScale(roi_gray, 1.1, 4)

  # 顔検出結果をバウンディングボックスで描画
  for (fx, fy, fw, fh) in faces:
    roi_face_gray = roi_gray[fy:fy+fh, fx:fx+fw]
    face_top_left = (x1 + fx, y1 + fy)
    face_bottom_right = (x1 + fx + fw, y1 + fy + fh)
    cv2.rectangle(frame, face_top_left, face_bottom_right, (0, 255, 0), 2)

    # 笑顔を検出・描画
    smiles = smile_cascade.detectMultiScale(roi_face_gray, 1.1, 15)
    for (sx, sy, sw, sh) in smiles:
      smile_top_left = (x1 + fx + sx, y1 + fy + sy)
      smile_bottom_right = (x1 + fx + sx + sw, y1 + fy + sy + sh)
      cv2.rectangle(frame, smile_top_left, smile_bottom_right, (255, 0, 0), 2)

  return frame

# モデルを設定
model = YOLO("yolov8n.pt")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml')

try:
  while cap.isOpened():
    # フレームを抽出
    ret, frame = cap.read()

    if ret:
      # 物体検出
      results = model(frame, verbose=False, classes=0)
      img_annotated = results[0].plot()

      # 分析（検出数をカウント）
      class_list = results[0].boxes.cls
      count_class = len(class_list)

      # 物体検出部分の領域を取得
      person_boxes = results[0].boxes.xyxy

      # 顔検出
      for box in person_boxes:
        img_annotated = detect_faces(img_annotated, box, face_cascade, smile_cascade)

      # 画像に検出数を表示
      img_annotated = text_overwrite_to_image(img_annotated, f"person={count_class}", (10, 25))
      colab_imshow("png", img_annotated)

except KeyboardInterrupt:
  print("中止しました")

In [ ]:
# @title (7) 笑顔の人数を表示 (本 6-5-3)

# VideoCaptureメソッドに渡す動画ファイル名は各自で入力

# 画像上部の人数表示の大きさを調整する場合、変える数値パラメータは下記3箇所
# extension_height = 40
# font_scale = 1
# img_annotated = text_overwrite_to_image(..., (10, 25))

# 顔・笑顔が検出されない(にくい)時は
# face(smile)_cascade.detectMultiScale のパラメータを調整 (本 p.200 参照)

from ultralytics import YOLO
import cv2
import numpy as np

def text_overwrite_to_image(img, text, position):
  """テキストを画像に描画する関数"""

  # 画像の横サイズを取得
  img_width = img.shape[1]

  # 黒い四角形（拡張領域）を作成
  extension_height = 40
  black_rectangle = np.zeros((extension_height, img_width, 3), dtype=np.uint8)

  # 元の画像と黒い四角形を結合
  extended_img = np.vstack((black_rectangle, img))

  # 文字の設定
  font = cv2.FONT_HERSHEY_SIMPLEX
  font_scale = 1
  font_color = (255, 255, 255)
  font_thickness = 2

  # 画像にテキストを上書き
  cv2.putText(extended_img,
              text=text,
              org=position,
              fontFace=font,
              fontScale=font_scale,
              color=font_color,
              thickness=font_thickness)

  return extended_img

def detect_faces(frame, person_box, face_cascade, smile_cascade):
  """顔と笑顔を検出・描画する関数"""

  # 物体検出領域から顔を検出
  x1, y1, x2, y2 = int(person_box[0]), int(person_box[1]), int(person_box[2]), int(person_box[3])
  roi_gray = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
  faces = face_cascade.detectMultiScale(roi_gray, 1.1, 4)

  # 顔検出結果をバウンディングボックスで描画
  is_smile = False
  for (fx, fy, fw, fh) in faces:
    roi_face_gray = roi_gray[fy:fy+fh, fx:fx+fw]
    face_top_left = (x1 + fx, y1 + fy)
    face_bottom_right = (x1 + fx + fw, y1 + fy + fh)
    cv2.rectangle(frame, face_top_left, face_bottom_right, (0, 255, 0), 2)

    # 笑顔を検出・描画
    smiles = smile_cascade.detectMultiScale(roi_face_gray, 1.1, 15)
    if len(smiles) > 0:
      is_smile = True
    for (sx, sy, sw, sh) in smiles:
      smile_top_left = (x1 + fx + sx, y1 + fy + sy)
      smile_bottom_right = (x1 + fx + sx + sw, y1 + fy + sy + sh)
      cv2.rectangle(frame, smile_top_left, smile_bottom_right, (255, 0, 0), 2)

  return frame, is_smile

# モデルを設定
model = YOLO("yolov8n.pt")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml')

cap = cv2.VideoCapture("17803_1280x720.mp4")

try:
  while cap.isOpened():
    # フレームを抽出
    ret, frame = cap.read()

    if ret:
      # 物体検出
      results = model(frame, verbose=False, classes=0)
      img_annotated = results[0].plot()

      # 分析（検出数をカウント）
      class_list = results[0].boxes.cls
      count_class = len(class_list)

      # 物体検出部分の領域を取得
      person_boxes = results[0].boxes.xyxy

      # 顔検出
      count_smile = 0
      for box in person_boxes:
        img_annotated, is_smile = detect_faces(img_annotated, box, face_cascade, smile_cascade)
        if is_smile:
          count_smile += 1

      # 画像を表示
      img_annotated = text_overwrite_to_image(img_annotated, f"person={count_class}", (10, 25))
      img_annotated = text_overwrite_to_image(img_annotated, f"smile={count_smile}", (10, 25))

      colab_imshow("png", img_annotated)

except KeyboardInterrupt:
  print("中止しました")

In [ ]:
# @title (8) 全員笑顔になったら撮影 (本 6-6-1)

# VideoCaptureメソッドに渡す動画ファイル名は各自で入力

# 画像上部の人数表示の大きさを調整する場合、変える数値パラメータは下記3箇所
# extension_height = 40
# font_scale = 1
# img_annotated = text_overwrite_to_image(..., (10, 25))

# 顔・笑顔が検出されない(にくい)時は
# face(smile)_cascade.detectMultiScale のパラメータを調整 (本 p.200 参照)

from ultralytics import YOLO
import cv2
import numpy as np
from pathlib import Path
import time

def text_overwrite_to_image(img, text, position):
  """テキストを画像に描画する関数"""

  # 画像の横サイズを取得
  img_width = img.shape[1]

  # 黒い四角形（拡張領域）を作成
  extension_height = 40
  black_rectangle = np.zeros((extension_height, img_width, 3), dtype=np.uint8)

  # 元の画像と黒い四角形を結合
  extended_img = np.vstack((black_rectangle, img))

  # 文字の設定
  font = cv2.FONT_HERSHEY_SIMPLEX
  font_scale = 1
  font_color = (255, 255, 255)
  font_thickness = 2

  # 画像にテキストを上書き
  cv2.putText(extended_img,
              text=text,
              org=position,
              fontFace=font,
              fontScale=font_scale,
              color=font_color,
              thickness=font_thickness)

  return extended_img

def detect_faces(frame, person_box, face_cascade, smile_cascade):
  """顔と笑顔を検出・描画する関数"""

  # 物体検出領域から顔を検出
  x1, y1, x2, y2 = int(person_box[0]), int(person_box[1]), int(person_box[2]), int(person_box[3])
  roi_gray = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
  faces = face_cascade.detectMultiScale(roi_gray, 1.1, 4)

  # 顔検出結果をバウンディングボックスで描画
  is_smile = False
  for (fx, fy, fw, fh) in faces:
    roi_face_gray = roi_gray[fy:fy+fh, fx:fx+fw]
    face_top_left = (x1 + fx, y1 + fy)
    face_bottom_right = (x1 + fx + fw, y1 + fy + fh)
    cv2.rectangle(frame, face_top_left, face_bottom_right, (0, 255, 0), 2)

    # 笑顔を検出・描画
    smiles = smile_cascade.detectMultiScale(roi_face_gray, 1.1, 15)
    if len(smiles) > 0:
      is_smile = True
    for (sx, sy, sw, sh) in smiles:
      smile_top_left = (x1 + fx + sx, y1 + fy + sy)
      smile_bottom_right = (x1 + fx + sx + sw, y1 + fy + sy + sh)
      cv2.rectangle(frame, smile_top_left, smile_bottom_right, (255, 0, 0), 2)

  return frame, is_smile

# モデルを設定
model = YOLO("yolov8n.pt")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml')

cap = cv2.VideoCapture("17803_1280x720.mp4")

# 画像保存の設定
file_counter = 0
last_saved_time = 0
folder = Path('img-smile')
folder.mkdir(exist_ok=True)

try:
  while cap.isOpened():
    # フレームを抽出
    ret, frame = cap.read()

    if ret:
      # 物体検出
      results = model(frame, verbose=False, classes=0)
      img_annotated = results[0].plot()

      # 分析（検出数をカウント）
      class_list = results[0].boxes.cls
      count_class = len(class_list)

      # 物体検出部分の領域を取得
      person_boxes = results[0].boxes.xyxy

      # 顔検出
      count_smile = 0
      for box in person_boxes:
        img_annotated, is_smile = detect_faces(img_annotated, box, face_cascade, smile_cascade)
        if is_smile:
          count_smile += 1

      # 画像を表示
      img_annotated = text_overwrite_to_image(img_annotated, f"person={count_class}", (10, 25))
      img_annotated = text_overwrite_to_image(img_annotated, f"smile={count_smile}", (10, 25))

      colab_imshow("png", img_annotated)

      # 人数と笑顔数が一致したら画像を保存
      current_time = time.time()
      if count_class == count_smile and (current_time - last_saved_time) > 10:
        filename = folder / f"image_{file_counter:04d}.jpg"
        cv2.imwrite(filename, frame)
        file_counter += 1
        last_saved_time = current_time

except KeyboardInterrupt:
  print("中止しました")

In [ ]:
# @title (9) 応用例：全員笑顔になったら動画を止めて撮影、文字と元画像 (少し大きく) を表示

# VideoCaptureメソッドに渡す動画ファイル名は各自で入力

# 画像上部の人数表示の大きさを調整する場合、変える数値パラメータは下記3箇所
# extension_height = 40
# font_scale = 1
# img_annotated = text_overwrite_to_image(..., (10, 25))

# 顔・笑顔が検出されない(にくい)時は
# face(smile)_cascade.detectMultiScale のパラメータを調整 (本 p.200 参照)

from ultralytics import YOLO
import cv2
import numpy as np
from pathlib import Path
import time

def text_overwrite_to_image(img, text, position):
  """テキストを画像に描画する関数"""

  # 画像の横サイズを取得
  img_width = img.shape[1]

  # 黒い四角形（拡張領域）を作成
  extension_height = 40
  black_rectangle = np.zeros((extension_height, img_width, 3), dtype=np.uint8)

  # 元の画像と黒い四角形を結合
  extended_img = np.vstack((black_rectangle, img))

  # 文字の設定
  font = cv2.FONT_HERSHEY_SIMPLEX
  font_scale = 1
  font_color = (255, 255, 255)
  font_thickness = 2

  # 画像にテキストを上書き
  cv2.putText(extended_img,
              text=text,
              org=position,
              fontFace=font,
              fontScale=font_scale,
              color=font_color,
              thickness=font_thickness)

  return extended_img

def detect_faces(frame, person_box, face_cascade, smile_cascade):
  """顔と笑顔を検出・描画する関数"""

  # 物体検出領域から顔を検出
  x1, y1, x2, y2 = int(person_box[0]), int(person_box[1]), int(person_box[2]), int(person_box[3])
  roi_gray = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
  faces = face_cascade.detectMultiScale(roi_gray, 1.1, 4)

  # 顔検出結果をバウンディングボックスで描画
  is_smile = False
  for (fx, fy, fw, fh) in faces:
    roi_face_gray = roi_gray[fy:fy+fh, fx:fx+fw]
    face_top_left = (x1 + fx, y1 + fy)
    face_bottom_right = (x1 + fx + fw, y1 + fy + fh)
    cv2.rectangle(frame, face_top_left, face_bottom_right, (0, 255, 0), 2)

    # 笑顔を検出・描画
    smiles = smile_cascade.detectMultiScale(roi_face_gray, 1.1, 15)
    if len(smiles) > 0:
      is_smile = True
    for (sx, sy, sw, sh) in smiles:
      smile_top_left = (x1 + fx + sx, y1 + fy + sy)
      smile_bottom_right = (x1 + fx + sx + sw, y1 + fy + sy + sh)
      cv2.rectangle(frame, smile_top_left, smile_bottom_right, (255, 0, 0), 2)

  return frame, is_smile

# モデルを設定
model = YOLO("yolov8n.pt")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml')

cap = cv2.VideoCapture("17803_1280x720.mp4")

# 画像保存の設定
file_counter = 0
last_saved_time = 0
folder = Path('img-smile')
folder.mkdir(exist_ok=True)

try:
  while cap.isOpened():
    # フレームを抽出
    ret, frame = cap.read()

    if ret:
      # 物体検出
      results = model(frame, verbose=False, classes=0)
      img_annotated = results[0].plot()

      # 分析（検出数をカウント）
      class_list = results[0].boxes.cls
      count_class = len(class_list)

      # 物体検出部分の領域を取得
      person_boxes = results[0].boxes.xyxy

      # 顔検出
      count_smile = 0
      for box in person_boxes:
        img_annotated, is_smile = detect_faces(img_annotated, box, face_cascade, smile_cascade)
        if is_smile:
          count_smile += 1

      # 画像を表示
      img_annotated = text_overwrite_to_image(img_annotated, f"person={count_class}", (10, 25))
      img_annotated = text_overwrite_to_image(img_annotated, f"smile={count_smile}", (10, 25))

      colab_imshow("png", img_annotated)

      # 人数と笑顔数が一致したら画像を保存
      current_time = time.time()
      if count_class == count_smile and (current_time - last_saved_time) > 10:
        filename = folder / f"image_{file_counter:04d}.jpg"
        cv2.imwrite(filename, frame)
        file_counter += 1
        last_saved_time = current_time
        print("全員笑顔☺️")
        colab_imshow("png", frame, 320)
        break

except KeyboardInterrupt:
  print("中止しました")

In [ ]:
# @title これで学習会は終わりです。お疲れ様でした！